# California Housing Linear Regression with Scikit-Learn

This project uses Scikit-Learn to preprocess numerical and categorical housing data, train a linear regression model, and evaluate its predictions.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


# 1. Load the dataset and separate the features from the target.
url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]


# 2. Split the data before fitting preprocessing steps.
# Stratification keeps the ocean_proximity categories represented across both sets.
X_Tr, X_Te, y_Tr, y_Te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=X["ocean_proximity"]
)


# 3. Separate categorical and numerical features because they need different preprocessing.
X_Tr_cat = X_Tr[["ocean_proximity"]]
X_Te_cat = X_Te[["ocean_proximity"]]

X_Tr_num = X_Tr.drop(columns=["ocean_proximity"])
X_Te_num = X_Te.drop(columns=["ocean_proximity"])


# 4. One-hot encode the categorical feature into numeric columns.
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_Tr_cat_E = encoder.fit_transform(X_Tr_cat)
X_Te_cat_E = encoder.transform(X_Te_cat)


# 5. Standardize numerical features using statistics learned from the training set.
scaler = StandardScaler()

X_Tr_num_S = scaler.fit_transform(X_Tr_num)
X_Te_num_S = scaler.transform(X_Te_num)


# 6. Fill missing numerical values using the five nearest neighbors.
imputer = KNNImputer(
    n_neighbors=5,
    weights="distance"
)

X_Tr_num_I = imputer.fit_transform(X_Tr_num_S)
X_Te_num_I = imputer.transform(X_Te_num_S)


# 7. Combine the processed numerical features with the encoded categorical features.
X_Tr_final = np.concatenate(
    [X_Tr_num_I, X_Tr_cat_E],
    axis=1
)

X_Te_final = np.concatenate(
    [X_Te_num_I, X_Te_cat_E],
    axis=1
)


# 8. Train linear regression, make predictions, and evaluate on the test set.
model = LinearRegression()

model.fit(X_Tr_final, y_Tr)


y_pred = model.predict(X_Te_final)


mse = mean_squared_error(y_Te, y_pred)
rmse = np.sqrt(mse)

print(f"MSE: {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")